In [58]:
import requests 
from dotenv import load_dotenv
import os 
load_dotenv()

#getting API key from env
api_key = os.getenv("COMPANIES_HOUSE_API_KEY")
if not api_key:
    print("COMPANIES_HOUSE_API_KEY is not set")

URL = "https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI"
#testing requests 

document_metadata = requests.get(URL, auth=(api_key,''))

print(document_metadata.status_code)



200


In [59]:
document_metadata.json()


{'company_number': '05150526',
 'barcode': 'XEXYI9JT',
 'significant_date': '2025-09-30T00:00:00Z',
 'significant_date_type': 'made-up-date',
 'category': 'accounts',
 'pages': 4,
 'filename': '05150526_aa_2026-03-16',
 'created_at': '2026-03-16T12:27:43.122466296Z',
 'etag': '',
 'links': {'self': 'https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI',
  'document': 'https://document-api.company-information.service.gov.uk/document/9N7sJvwmt0UtuWyTEEZ4q3BRBLLktpAuRGa4ZCxRSxI/content'},
 'resources': {'application/pdf': {'content_length': 30752},
  'application/xhtml+xml': {'content_length': 34960}}}

In [61]:
link_document = document_metadata.json()['links']['document']
headers = {
    "Accept": "application/xhtml+xml"
}

document_response = requests.get(link_document, 
                                 auth=(api_key,''),
                                 headers = headers
)


print(document_response.status_code)

document_response.headers['Content-Type']


200


'application/xhtml+xml'

In [85]:
from bs4 import BeautifulSoup
document_soup = BeautifulSoup(document_response.text, 'html.parser')
document_tag = document_soup.find("ix:nonfraction")

#find the current assets
current_asset_tag = document_soup.find(
    "ix:nonfraction",
    attrs={"name": "frs-core:CurrentAssets"}
           )
current_asset_tag.text                    


'1,321'

In [86]:
document_tag["contextref"]

'CURRENT_FY_END'

In [87]:
soup_context = document_soup.find("xbrli:context",id='CURRENT_FY_END')
soup_context

<xbrli:context id="CURRENT_FY_END">
<xbrli:entity>
<xbrli:identifier scheme="http://www.companieshouse.gov.uk/">05150526</xbrli:identifier>
</xbrli:entity>
<xbrli:period>
<xbrli:instant>2025-09-30</xbrli:instant>
</xbrli:period>
</xbrli:context>

In [83]:
for tag in document_soup.find_all("ix:nonfraction"):
    if "1,321" in tag.text:
        print(tag)

<ix:nonfraction contextref="CURRENT_FY_END" decimals="0" format="ixt:numcommadot" name="frs-core:CurrentAssets" unitref="GBP">1,321</ix:nonfraction>


In [121]:
 #get filing history of scci

company_number = "05150526"

filing_history_url = f"https://api.company-information.service.gov.uk/company/{company_number}/filing-history"

filing_history = requests.get(filing_history_url, auth=(api_key,''))
filing_history_document = filing_history.json()
filing_history_document['items'][9]['description']

'confirmation-statement-with-no-updates'

In [137]:

def get_full_account(filing_history_document):
    filing = []
    for filings in filing_history_document['items']:
        
    
        if filings['description'] == 'accounts-with-accounts-type-total-exemption-full':
            filing = filings
            return filing

    
scci_filling_full_account = get_full_account(filing_history_document)
scci_full_account_link = scci_filling_full_account['links']['document_metadata']


scci_full_accounts_response = requests.get(scci_full_account_link, auth=(api_key,''))
print(scci_full_accounts_response.status_code)
scci_full_accounts_response.json()

200


{'company_number': '05150526',
 'barcode': 'X9YDNAWQ',
 'significant_date': '2020-03-31T00:00:00Z',
 'significant_date_type': 'made-up-date',
 'category': 'accounts',
 'pages': 10,
 'filename': '',
 'created_at': '2021-02-15T16:11:04.084504912Z',
 'etag': '',
 'links': {'self': 'https://document-api.company-information.service.gov.uk/document/vBRLLhySgOGunl2dfkHTRblkVonL6vmNfBsfSHlmIu4',
  'document': 'https://document-api.company-information.service.gov.uk/document/vBRLLhySgOGunl2dfkHTRblkVonL6vmNfBsfSHlmIu4/content'},
 'resources': {'application/pdf': {'content_length': 150442},
  'application/xhtml+xml': {'content_length': 143987}}}